# **Herbal**

#### 1. Importing requried lib

In [2]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU Name: NVIDIA GeForce GTX 1650 Ti


In [1]:
import numpy
import torch
import transformers
import sentence_transformers
import faiss
import langchain

print("NumPy:", numpy.__version__)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence-Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("LangChain:", langchain.__version__)

c:\Users\yedee\anaconda3\envs\conda_genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 2.2.6
Torch: 2.5.1+cu121
Transformers: 5.5.0
Sentence-Transformers: 5.3.0
FAISS: 1.13.2
LangChain: 1.2.15


In [4]:
import langchain
print(langchain.__version__)  # Should be 0.1.0 or higher


1.2.15


In [6]:
import langchain
from IPython.display import Markdown
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import HumanMessagePromptTemplate,SystemMessagePromptTemplate,AIMessagePromptTemplate,PromptTemplate
from langchain_huggingface import ChatHuggingFace,HuggingFaceEmbeddings
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate,HumanMessagePromptTemplate,MessagesPlaceholder,SystemMessagePromptTemplate
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.tools import WikipediaQueryRun,DuckDuckGoSearchResults
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent
from langchain_community.agent_toolkits.openapi.toolkit import RequestsToolkit
from langchain_community.utilities.requests import TextRequestsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI


from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_core.tools.retriever import create_retriever_tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

import os
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

#### 2. Load Model

In [8]:
groq = ChatGroq(model = 'llama-3.1-8b-instant')
groq

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001AD1AFA5C30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AD1AFA5C90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### 3. Load Documents - Pypdf

In [12]:
import os
files = []

for file in os.listdir('data'):
    if file.lower().endswith('.pdf'):
        files.append(os.path.join('data',file))

docs = []

for file in files:
    loader = PyPDFLoader(file)
    loaded_docs = loader.load()
    docs.extend(loaded_docs)


KeyboardInterrupt: decoding with 'utf-16-be' codec failed (KeyboardInterrupt: )

**loaded_docs = loader.load()**
- Reads the PDF and loads its content.
- Usually, each page of the PDF is stored as a separate Document object.

In [7]:
len(docs)

2092

In [8]:
type(docs)

list

#### 4. Test spliting

- **CharacterTextSplitter** — Splits text into fixed-size chunks based purely on character count.
- **RecursiveCharacterTextSplitter** — Recursively splits text using separators (paragraph → sentence → word) to preserve context.
- **TokenTextSplitter** — Splits text based on token count to match LLM input limits.
- **NLTKTextSplitter** — Splits text into chunks using sentence boundaries via NLTK.
- **SpacyTextSplitte**r — Uses SpaCy NLP to split text into linguistically meaningful sentences.
- **MarkdownHeaderTextSplitter** — Splits text based on Markdown headers to preserve document structure.
- **HTMLHeaderTextSplitter** — Divides text using HTML tags and structure.
- **SemanticChunker** — Splits text based on semantic meaning using embeddings similarity.
- **SentenceTransformersTokenTextSplitte**r — Combines sentence-transformers with token limits for better semantic chunking.
- **LatexTextSplitter** — Splits LaTeX documents based on sections and formatting.

In [9]:
# creating the text spliter
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 300,
    separators=["\n\n", "\n", ".", " ", ""]
)

pdf_chuncks = text_spliter.split_documents(docs)

print(pdf_chuncks[0].page_content)

Journal of Pre-Clinical and Clinical Research, 2014, Vol 8, No 2, 55-60
www.jpccr.euREVIEW  ARTICLE 
Herbal medicine for treatment and prevention 
of liver diseases
Mayuresh Rajaratnam1, Andrzej Prystupa2, Patrycja Lachowska-Kotowska2, Wojciech Załuska3, 
Rafał Filip4
1 Research Students’ Association, Medical University, Lublin, Poland  
2 Department of Internal Medicine, Medical University, Lublin, Poland  
3 Department of Nephrology, Medical University, Lublin, Poland  
4 Institute of Rural Health, Lublin, Poland
Rajaratnam M, Prystupa A, Lachowska-Kotowska P , Załuska W, Filip R. Herbal medicine for treatment and prevention of liver diseases. J Pre-
Clin Clin Res. 2014; 8(2): 55–60. doi: 10.5604/18982395.1135650
Abstract
The rising number of patients with liver dysfunction due to overwhelming usage of drugs and alcohol has paved the path 
for researchers in an interest in herbal medicine. This is because there are only a few universally effective and available 
options for the treat

- Advantages:
- Preserves paragraphs → sentences → words
- Best for RAG pipelines

- Most recommended in real projects

In [10]:
print(pdf_chuncks[0].page_content)

print("total chunks :", len(pdf_chuncks))
print()
print("first 5 chunks")
print()

for i in range(5):
    print(f"chunk {i+1}")
    print(pdf_chuncks[i].page_content)
    print("-" * 100)

Journal of Pre-Clinical and Clinical Research, 2014, Vol 8, No 2, 55-60
www.jpccr.euREVIEW  ARTICLE 
Herbal medicine for treatment and prevention 
of liver diseases
Mayuresh Rajaratnam1, Andrzej Prystupa2, Patrycja Lachowska-Kotowska2, Wojciech Załuska3, 
Rafał Filip4
1 Research Students’ Association, Medical University, Lublin, Poland  
2 Department of Internal Medicine, Medical University, Lublin, Poland  
3 Department of Nephrology, Medical University, Lublin, Poland  
4 Institute of Rural Health, Lublin, Poland
Rajaratnam M, Prystupa A, Lachowska-Kotowska P , Załuska W, Filip R. Herbal medicine for treatment and prevention of liver diseases. J Pre-
Clin Clin Res. 2014; 8(2): 55–60. doi: 10.5604/18982395.1135650
Abstract
The rising number of patients with liver dysfunction due to overwhelming usage of drugs and alcohol has paved the path 
for researchers in an interest in herbal medicine. This is because there are only a few universally effective and available 
options for the treat

#### 5. Creating the embeddings
- Hugging Face mode

In [9]:
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("gpu name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

device: cuda
cuda available: True
cuda version: 12.1
gpu name: NVIDIA GeForce GTX 1650 Ti


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

hf_embedding = HuggingFaceEmbeddings(
    model_name = 'BAAI/bge-large-en-v1.5',
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs = {'batch_size':16}

)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1233.86it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### 6. Creating the database Faiss

In [135]:
vector_db_path = 'faiss_law_db'

if os.path.exists(vector_db_path):
    vector_store = FAISS.load_local(
        vector_db_path,
        hf_embedding,
        allow_dangerous_deserialization=True
    )
    print("vector db loaded successfully using local saved data")
else:
    vector_store = FAISS.from_documents(pdf_chuncks,hf_embedding)
    vector_store.save_local(vector_db_path)
    print("No Database was found vector db created and saved successfully")

vector db loaded successfully using local saved data


#### 7. Creating the retiver

In [136]:
retriver = vector_store.as_retriever(
    search_type = 'mmr',
    search_kwargs = {'k':5}
)

- **similarity** — Returns top-k documents based on highest vector similarity to the query.
- **similarity_score_threshold** — Returns documents whose similarity score is above a defined threshold.
- **mmr (Max Marginal Relevance)** — Retrieves diverse and relevant documents by balancing similarity and diversity.

In [137]:
question = 'what is Punarnava'

- Extract(retriver) the answer from database

In [138]:
retrieved_docs = retriver.invoke(question)
retrieved_docs

[Document(id='c13c563a-76b4-460e-9e44-7c6963f7ac8e', metadata={'producer': 'Adobe PDF Scan Library 4.1', 'creator': 'Adobe Scan-to-PDF Utility 4.0', 'creationdate': '2014-01-27T18:08:52+05:30', 'keywords': 'converted', 'moddate': '2014-01-27T18:08:56+05:30', 'source': 'data\\101-(MEDICINAL.pdf', 'total_pages': 73, 'page': 20, 'page_label': '21'}, page_content="Medicinal Plants in the Indian Arid Zone ~~ , \nBoerhavia diffusa Linn. (Nyctaginaceae) \nEnglish \nSanskrit \nHindi \nLocal \nHogweed \nPunarnava \nSanthi \nSanti, crunawari \n~-----------------------, \nWild all over desertic area in slightly sandy soil or around dumping places, nov. \ncultivated also. A deep rooted spreading perennial herb, with a stout root-stock \nand many erect or spread ing branches. Stem is creeping often purplish. Th(; \nleaves are simple, thick and brittle usually whitish and smooth beneath and rough \ngreen on upper surface. Flowers white or pink in long stalked umbels. Fruits \noval. 5-ribbed and gree

In [139]:
retrieved_docs = retriver.invoke(question)

from IPython.display import Markdown

formatted_output = ""

for i, doc in enumerate(retrieved_docs, 1):
    formatted_output += f"### 📄 Result {i}\n\n"
    formatted_output += f"{doc.page_content}\n\n"
    formatted_output += "---\n\n"

Markdown(formatted_output)

### 📄 Result 1

Medicinal Plants in the Indian Arid Zone ~~ , 
Boerhavia diffusa Linn. (Nyctaginaceae) 
English 
Sanskrit 
Hindi 
Local 
Hogweed 
Punarnava 
Santhi 
Santi, crunawari 
~-----------------------, 
Wild all over desertic area in slightly sandy soil or around dumping places, nov. 
cultivated also. A deep rooted spreading perennial herb, with a stout root-stock 
and many erect or spread ing branches. Stem is creeping often purplish. Th(; 
leaves are simple, thick and brittle usually whitish and smooth beneath and rough 
green on upper surface. Flowers white or pink in long stalked umbels. Fruits 
oval. 5-ribbed and green or brown in colour. Flowering and fruiting in through 
out the year, but maximum in rainy season. 
Part used: Whole plant 
Usage: The whole plant fresh or dried, is the source of drug 'Punamava'. it is diuretic, antipyretic and laxati ve. 
Fresh boiled herb or liquid extract of the fresh or dry plant is very effective in the treatment of dropsy, 
inflammatory renal diseases and ascites. Fresh juice given as a blood purifier and to get relief from muscular pain. 
The leaves eaten as vegetable to cure night blindness . 
Root powder effective in stomach disorder, particularly intestinal colic and in expelling 
intestinal worms, curing asthma, kidney and heart problem, gonorrhea, anemia, cough, nervous 
weakness and paralysis. The roots are considered useful for jaundice. A paste of the root can be 
applied in several skin diseases.

---

### 📄 Result 2

fluid or blood from the body forcibly, and ‗Da‘ 
suggests donating [1]. Ayurveda is an ancient and 
most ordinarily practiced type of medicine in 
India. Ayurveda comes from the words Ayur (life) 
and religious text (knowledge) [2]. Cardiopathy 
(Hridroga) may be an international phenomenon. It 
is currently turning into a serious unhealthiness 
even in developing countries. The international 
burden of disease is calculable at concerning 272 
death per large integer population and mortality is 
inflated by 59% in India [3]. Hridayroga or vas 
diseases include heart attacks, angina pecto ris, 
hypertension, innate cardiopathy, coronary heart 
illness, and plenty more. Written material offers 
numerous herbs and preventive ways for treating 
these heart diseases. Merely saying, Rasa (body 
fluids) and Rakta (blood) is circulated within the 
body, by the twin action of physical assortment 
and therefore the name Hrudaya in Ayurveda. This 
is often the elemental function of the heart. In line 
with Ayurvedic texts, the heart originates from the 
essence of Rakta and Kapha, preponderantly from 
the matern al side, and develops into a muscular 
organ [1]. 
Ayurveda is a holistic discipline that can 
improve heart health and even offer a cure for 
heart disease. The heart is one of the major vital 
organs that regulate blood flow and control the 
normal function of  the body. In Ayurveda heart is 
known as Hriday and any damage to the heart

---

### 📄 Result 3

Nervinetas; Plantival. Portugal: Neurocardol; Songha; V ale-
sono. Russia: Doppelherz Vitalotonik ( Доппельгерц Витало-
тоник); Herbion Drops for the Heart ( Гербион Сердечные
Капли); Persen ( Персен); Sanason ( Санасон). South Africa:
Avena Sativa Comp; Biral; Entressdruppels HM; Helmont-
skruie; Krampdruppels; Restin; Stuidruppels; Wonderkroones-
sens. Spain: Natusor Somnisedan; Nervikan; Relana; Sedasor;
Sedonat; V aldispert Complex. Switzerland: Baldriparan; Dor-
measan; Dormiplant; Dragees pour la detente nerveuse;
Dragees pour le coeur et les nerfs; Dragees pour le sommeil;
Dragees sedatives Dr Welti; Hova; Nervinetten; ReDormin;
Relaxane; Relaxo; Songha Night; Soporin; Strath Gouttes pour
le nerfs et contre l'insomnie; Tisane calmante pour les enfants;
Tisane pour le sommeil et les nerfs; V alverde Coeur; V alverde
Detente dragees; V alverde Sommeil; V alviska; Zeller Sommeil.
UK: Avena Sativa Comp; Bio-Strath V alerian Formula; Boots
Alternatives Sleep Well; Boots Sleepeaze Herbal Tablets;
Calmanite Tablets; Constipation T ablets; Daily T ension &
Strain Relief; Digestive; Fenneherb Newrelax; Fenneherb
Prementaid; Gerard House Serenity; Gerard House Somnus;
Golden Seal Indigestion Tablets; Herbal Indigestion Natur-
tabs; Herbal Pain Relief; HRI Calm Life; HRI Golden Seal
Digestive; HRI Night; Indigestion and Flatulence; Kalms Sleep;
Kalms; Laxative T ablets; Menopause Relief; Modern Herbals
Stress; Napiers Digestion T ablets; Napiers Sleep Tablets;

---

### 📄 Result 4

J. Res. Trad. Medicine  Nov-Dec 2015  Volume 1  Issue 1| | |
4
The present study was conducted with an aim to 
identify the herbs mentioned in the Telugu cantos of 
Basvarajeeyam and to re-identify the specimen through 
eld study followed by botanical identication in 
consultation with units of CCRAS and Botanical Survey 
of India. The herbs that have been described in Telugu 
verses of Basavarajeeyam are considered to be an elite 
contribution to the Indian system of medicine. Target of 
study also includes collection of information regarding 
botanical source, distribution and important available 
local knowledge of the medicinal values of herbs which 
were newly introduced by Basavaraju.
Materials & Methods
As a rst step, the entire text of Basavarajeeyam printed 
by Puvvada (1929) taken up for the current study was 
analysed and scrutinized to enumerate and tabulate the 
herbs. The data obtained from this conceptual analysis 
was used for the further eld survey. Field survey was 
carried out in the areas of Mehaboobnagar, Nalgonda, 
Bellari, Anatapura, Kaddapa, Kurnool and Srikakulam. 
Field visit was planned to verify the vernacular name of 
plants given in the text. The author mentions that he 
had belonged to Koturu village and several villages 
with this name are present in the districts adjoining 
Karnataka state. The Telugu verse mentioned in the 
book are in vogue in Telangana and bordering districts 
of Karnataka region (Bellari, Anantapur, Cutappa and

---

### 📄 Result 5

Phyllanthus niruri  also referred to as "Bhumi Amla ," is a 
plant that contains flavonoids, tannins and lignans 
(phyllanthin and hypophyllanthin). Antiviral action 
(particularly against the hepatitis B virus), antioxidant 
defense and prevention of hepatic stellate cell activation are 
the main mechanisms by which it exerts its hepatoprotective 
effect. It is frequently used to treat hepatitis and jaundice in 
Ayurvedic medicine.15 
3.3. Andrographis paniculata 
Known as "Kalmegh," this plant contains a lot of 
andrographolide, a diterpenoid lactone that has strong anti -
inflammatory and hepatoprotective effects. It protects 
hepatocytes from damage caused by toxins by suppressing 
pro-inflammatory mediators including TNF -α and IL -1β. 
Additionally, it strengthens natural antioxidant enzymes like 
catalase and SOD.16 
3.4. Boerhavia diffusa 
It is used in Ayurveda to treat hepatic failure and ascites and 
is called "Punarnava." Its diuretic and anti -inflammatory 
qualities are attributed to the presence of punarnavine, 
alkaloids and flavonoids. B. Diffusion helps the regulation of 
bile flow, low ers high liver enzymes and stabilizes hepatic 
membranes.17 
3.5. Picrorhiza kurroa 
Picroside I and II, found in P. kurroa , an endangered 
Himalayan he rb, have potent immunomodulatory and 
hepatoprotective properties. In experimental studies, these 
iridoid glycosides prevent hepatotoxicity from paracetamol 
and CCl4, reduce oxidative stress and suppress inflammatory

---



In [140]:
retrieved_docs = retriver.invoke(question)

context = "\n\n".join([doc.page_content for doc in retrieved_docs])

print(context)

Medicinal Plants in the Indian Arid Zone ~~ , 
Boerhavia diffusa Linn. (Nyctaginaceae) 
English 
Sanskrit 
Hindi 
Local 
Hogweed 
Punarnava 
Santhi 
Santi, crunawari 
~-----------------------, 
Wild all over desertic area in slightly sandy soil or around dumping places, nov. 
cultivated also. A deep rooted spreading perennial herb, with a stout root-stock 
and many erect or spread ing branches. Stem is creeping often purplish. Th(; 
leaves are simple, thick and brittle usually whitish and smooth beneath and rough 
green on upper surface. Flowers white or pink in long stalked umbels. Fruits 
oval. 5-ribbed and green or brown in colour. Flowering and fruiting in through 
out the year, but maximum in rainy season. 
Part used: Whole plant 
Usage: The whole plant fresh or dried, is the source of drug 'Punamava'. it is diuretic, antipyretic and laxati ve. 
Fresh boiled herb or liquid extract of the fresh or dry plant is very effective in the treatment of dropsy, 
inflammatory renal diseases 

#### 8. Metrics

- (question = "what is Punarnava"), you should evaluate:

1. Retrieval Metrics
Context Precision → Are retrieved docs relevant?
Context Recall → Did we retrieve all useful info?
2. Generation Metrics
Answer Relevancy
Faithfulness (Groundedness) → Is answer based on context?

##### 1. Define Ground Truth (manual for now)

In [141]:
#ground_truth = "Punarnava is a medicinal herb used in Ayurveda for kidney and liver disorders"

In [142]:
ground_truths = [
    "Punarnava is a medicinal herb",
    "Punarnava is an Ayurvedic herb",
    "Punarnava is a plant used in Ayurveda"
]

##### 2. Generate Answer (use your LLM)

In [154]:
prompt = f"""
Answer the question clearly using the context.

Give a short definition.

Context:
{context}

Question:
{question}

Answer:
"""

response = groq.invoke(prompt)
answer = response.content
print("Generated Answer:\n", answer)

Generated Answer:
 Punarnava is a medicinal plant, specifically Boerhavia diffusa Linn., used in Ayurveda to treat various health conditions, including hepatic failure, ascites, and inflammatory renal diseases. It is also known as Hogweed, Santhi, or Santi in different languages.


##### RETRIEVAL METRICS

##### 3. Mark relevant docs manually (basic method)

In [155]:
relevant_docs = []

for doc in retrieved_docs:
    if any(word in doc.page_content.lower() for word in ["punarnava","herb","ayurveda"]):
        relevant_docs.append(doc)

##### 4. Precision

In [156]:
precision = len(relevant_docs) / len(retrieved_docs)
precision

1.0

##### 5. Recall (assume total relevant = 5 for example)

In [157]:
total_relevant_docs = 5   # you define manually

recall = len(relevant_docs) / total_relevant_docs
recall

1.0

##### GENERATION METRICS

##### 6. Token Overlap (simple method)

In [158]:
gt_tokens = set(clean(ground_truth))
ans_tokens = set(clean(answer))

gt_tokens = gt_tokens - stopwords
ans_tokens = ans_tokens - stopwords

common_tokens = gt_tokens.intersection(ans_tokens)
common_tokens

{'ayurveda', 'medicinal', 'punarnava'}

##### 7. Precision (Generation)

In [159]:
gen_precision = len(common_tokens) / len(ans_tokens)
gen_precision

0.10344827586206896

##### 8. Recall (Generation)

In [160]:
gen_recall = len(common_tokens) / len(gt_tokens)
gen_recall

0.42857142857142855

##### 9. F1 Score

In [161]:
f1 = (2 * gen_precision * gen_recall) / (gen_precision + gen_recall + 1e-10)
f1

0.16666666663533947

##### 10. Faithfulness (simple check)

In [162]:
supported = 0

for token in common_tokens:
    if token in context.lower():
        supported += 1

faithfulness = supported / (len(common_tokens) + 1e-10)

##### 11. FINAL RAG SCORE

In [163]:
rag_score = (precision + recall + f1 + faithfulness) / 4
rag_score

0.7916666666505016

##### 12. all Metrics

In [164]:
print("===== RAG METRICS =====")
print("Retrieval Precision :", round(precision,3))
print("Retrieval Recall    :", round(recall,3))
print("Generation F1       :", round(f1,3))
print("Faithfulness        :", round(faithfulness,3))
print("RAG Accuracy        :", round(rag_score,3))

===== RAG METRICS =====
Retrieval Precision : 1.0
Retrieval Recall    : 1.0
Generation F1       : 0.167
Faithfulness        : 1.0
RAG Accuracy        : 0.792



- Retrieval Precision = 1.0
- Retrieval Recall = 1.0
- Generation F1 = ~0.16
- Faithfulness = 1.0
- RAG Accuracy ≈ 0.79

- RAG Accuracy ≈ 0.79 -> GOOD

- Retrieval -> Excellent
- Faithfulness -> Excellent
- Generation -> Metric limitation
- Overall -> Strong system

#### 9. Pass Rag To LLM

##### 1: Create Prompt Template

In [182]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using the given context.

Give STRICT output in this format:

1. Introduction (2 lines ONLY)

2. Uses (ONLY 3 points)
- ...
- ...
- ...

3. Diseases / Conditions (ONLY 3 points)
- Disease: short explanation
- Disease: short explanation
- Disease: short explanation

4. Advantages (ONLY 3 points)
- ...
- ...
- ...

Additional Rule:
- If the question is about medicines for kidney problems or uric acid and like heart,liver,skin,brain etc:
  - Include specific medicine names or plant/herbal names (from the context)
  - Clearly mention them in Uses or Diseases section

Rules:
- Do NOT exceed number of points
- Keep answers short and precise
- Use only important information
- Do not include unnecessary details

Context:
{context}

Question:
{input}

Answer:
""")

##### 2: Create Document Chain

In [178]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(groq,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question using the given context.\n\nGive STRICT output in this format:\n\n1. Introduction (2 lines ONLY)\n\n2. Uses (ONLY 3 points)\n- ...\n- ...\n- ...\n\n3. Diseases / Conditions (ONLY 3 points)\n- Disease: short explanation\n- Disease: short explanation\n- Disease: short explanation\n\n4. Advantages (ONLY 3 points)\n- ...\n- ...\n- ...\n\nRules:\n- Do NOT exceed number of points\n- Keep answers short and precise\n- Use only important information\n- Do not include unnecessary details\n\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n'), additional_

##### 3: Create Retrieval Chain (MAIN STEP)

In [179]:
from langchain_classic.chains.retrieval import create_retrieval_chain

rag_chain = create_retrieval_chain(retriver,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001AD87D63520>, search_type='mmr', search_kwargs={'k': 5}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question using the given context.\n\nGive STRICT output in this format:\n\n1. Introducti

##### 4: Run RAG (Final)

In [180]:
response = rag_chain.invoke({"input": question})

print(response["answer"])

1. Introduction
Punarnava is a medicinal plant used in Ayurveda.

2. Uses
- Diuretic
- Antipyretic
- Laxative
- Treatment of dropsy
- Inflammatory renal diseases
- Ascites

3. Diseases / Conditions
- Dropsy: a condition characterized by swelling due to fluid accumulation in the body
- Inflammatory renal diseases: kidney diseases caused by inflammation
- Ascites: a condition where fluid accumulates in the abdominal cavity

4. Advantages
- Effective in treating various health conditions
- Has diuretic, antipyretic, and laxative properties
- Can be used to treat various diseases and conditions


#### 10. Chat type

In [ ]:
# CHAT LOOP

print(" Herbal Asistant Assistant")
print("Type 'exit' to quit\n")

while True:

    question = input("You: ")

    if question.lower() == "exit":
        break

    # run rag chain directly
    response = rag_chain.invoke({
        "input": question
    })

    print(" AI Lawyer:\n")
    print(response["answer"])
    print()

 AI Lawyer — Indian Law Assistant
Type 'exit' to quit

 AI Lawyer:

1. Introduction
- Gout is a type of kidney disease characterized by elevated levels of uric acid in the blood.
- It can lead to the formation of kidney stones and other complications.

2. Uses
- Treats kidney stones
- Relieves bladder inflammation
- Has antiseptic properties to reduce bacteria populations in the kidneys and urinary tract

3. Diseases / Conditions
- Gout: a type of kidney disease characterized by elevated levels of uric acid in the blood
- Kidney stones: hard deposits that form in the kidneys and can cause severe pain
- Nephritis: inflammation of the kidneys that can be caused by various factors, including infections and autoimmune disorders

4. Advantages
- Bearberry leaves have diuretic properties that increase urine volume
- They have antiseptic properties that reduce bacteria populations in the kidneys and urinary tract
- They relieve bladder inflammation and help relieve the pain of kidney stones

